# 3x3 FaultBandNet training for 3DAR_EV (Whine / PWM / Wind)

In [ ]:
import csv, json, time, random, hashlib, os
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import soundfile as sf
from scipy import signal
from sklearn.metrics import roc_auc_score
import torch
from torch import nn
import torch.nn.functional as F

# ==========================================
# 1. Configuration
# ==========================================
CONFIG = {
    # --- 경로 ---
    'data_root': '/home/kos/Desktop/3DAR_EV/[3DAR Electric Car]/3DAR_EV',
    'out_dir': 'runs/nb_faultbandnet_all_20261124',
    'feature_cache_dir': 'runs/nb_faultbandnet_20261124',
    'device': 'cuda', 'train': True, 'workers': 10,
    # --- 녹음 / 위치 ---
    'sample_rate': 48000, 'positions': ['FRONT', 'REAR', 'RIGHT'], 'fault_types': ['Whine', 'PWM', 'Wind'],
    # --- 특징: 고장 대역 × 마이크 각도 지도 ---
    'n_fft': 8192, 'hop_length': 480, 'angle_bins': 36,
    'wind_band': [3300, 6200], 'wind_bin_hz': 50, 'wind_flanks': [4000, 5500],
    'pwm_band': [4800, 5200], 'pwm_bins': 68, 'pwm_line_exclude': [4975, 5025],
    'whine_order': 16, 'whine_halfwidth_bins': 25,
    'rpm_refine_orders': [4, 8, 12, 16],
    # --- 학습 ---
    'exclude_classes': [],
    'healthy_like': ['Imbalance', 'Modulation'],
    'output_mode': '3x3',
    'repeats': 30, 'test_rate': 0.053, 'split_seed': 0,
    'validation_fraction': 0.15, 'seed_base': 1000,
    'batch_size': 32, 'epochs': 80, 'patience': 15, 'learning_rate': 1e-3, 'weight_decay': 1e-3,
    'max_pos_weight': 20.0, 'threshold': 0.5, 'gpu_duty': 0.5,
    'log_every': 10,
    # --- 파인튜닝 ---
    'finetune': {
        'init_run_dir': None,
        'init_output_mode': None,
        'freeze': [],
        'learning_rate': None,
        'epochs': None,
    },
}
POS, TY = CONFIG['positions'], CONFIG['fault_types']
OUT = Path(CONFIG['out_dir']); OUT.mkdir(parents=True, exist_ok=True)

# ==========================================
# 0. 보조 함수
# ==========================================
def read_csv(path):
    with open(path, encoding='utf-8-sig', newline='') as f: return list(csv.DictReader(f))

def write_csv(path, rows):
    with open(path, 'w', encoding='utf-8-sig', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

def write_json(path, obj): Path(path).write_text(json.dumps(obj, ensure_ascii=False, indent=1), encoding='utf-8')

def seed_all(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

class GpuThrottle:
    def __init__(self, duty): self.duty = duty; self.t = time.perf_counter()
    def reset(self): self.t = time.perf_counter()
    def step(self):
        if self.duty >= 1: return
        if torch.cuda.is_available(): torch.cuda.synchronize()
        busy = time.perf_counter() - self.t; time.sleep(busy * (1 - self.duty) / max(self.duty, .05)); self.t = time.perf_counter()

# ==========================================
# 2. Parsing Dataset
# ==========================================
def discover(root):
    root = Path(root); recs = []
    for row in read_csv(root / '측정_목록.csv'):
        if int(row['스피커수']) != 3: continue
        p = {k: root / str(row[c]).replace('\\', '/') for k, c in [('wav', 'WAV'), ('info', '실험정보CSV'), ('angles', '각도CSV')]}
        info = {s['Speaker_Loc']: s for s in read_csv(p['info'])}
        if set(info) != set(POS) or any(s['status'] != 'OK' for s in info.values()): raise ValueError('Bad capture: ' + row['측정ID'])
        cls = [info[q]['Sound_Class'] for q in POS]
        recs.append(dict(uid=row['측정ID'], wav=str(p['wav']), angles=str(p['angles']), classes=cls,
                         fault_count=sum(c != 'Healthy' for c in cls), date=info[POS[0]]['event_time'][5:10],
                         source_files=[info[q]['Source_File'].replace('\\', '/') for q in POS],
                         rpm_intervals=[list(map(float, info[q]['Estimated_RPM_Interval'].split('~'))) for q in POS]))
    recs.sort(key=lambda r: r['uid']); return recs

# ==========================================
# 3. Feature Extraction
# ==========================================
def angle_track(path, duration):
    rows = read_csv(path); t = np.array([float(r['elapsed_sec']) for r in rows]); a = np.array([float(r['angle_deg']) for r in rows])
    keep = np.isfinite(t) & np.isfinite(a) & (t >= 0) & (t <= duration); t, a = t[keep], a[keep]
    t, u = np.unique(t, return_index=True); a = a[u]
    inc = np.r_[True, a[1:] > np.maximum.accumulate(a[:-1])]; t, a = t[inc], a[inc]
    k = (t > 0) & (t < duration) & (a > 0) & (a < 360)
    return np.r_[0.0, t[k], duration], np.r_[0.0, a[k], 360.0]

def make_rpm(r0, r1, c, dur):
    return lambda t: (lambda u: r0 + (r1 - r0) * u + c * (r1 - r0) * u * (1 - u))(np.clip(t / dur, 0, 1))

def order_spectrum(x, fs, rpm_fn, spr=512):
    b, a = signal.butter(6, 10000, fs=fs); x = signal.filtfilt(b, a, x.astype(np.float64)); t = np.arange(len(x)) / fs
    rev = np.concatenate([[0], np.cumsum(rpm_fn(t[:-1]) / 60.0 / fs)]); m = int(rev[-1] * spr)
    xr = np.interp(np.arange(m) / spr, rev, x); P = np.abs(np.fft.rfft(xr * np.hanning(m))) ** 2
    return np.fft.rfftfreq(m, 1.0 / spr), 10 * np.log10(P + 1e-30)

def peak_over_bg(o, P, c, half=0.02, bg=(0.15, 0.6)):
    d = np.abs(o - c); return float(P[d <= half].max() - np.median(P[(d >= bg[0]) & (d <= bg[1])]))

def refine_rpm(audio, fs, r0, r1, dur):
    def sharp(q):
        o, P = order_spectrum(audio, fs, make_rpm(r0 * (1 + q[0]), r1 * (1 + q[1]), q[2], dur))
        return float(np.mean([peak_over_bg(o, P, k) for k in CONFIG['rpm_refine_orders']]))
    p = np.zeros(3); best = sharp(p)
    for step in (0.01, 0.0025):
        improved = True
        while improved:
            improved = False
            for j in range(3):
                for d in (-step, step):
                    q = p.copy(); q[j] += d * (10 if j == 2 else 1); v = sharp(q)
                    if v > best + 1e-3: best, p, improved = v, q, True
    return p

def extract_maps(rec):
    c = CONFIG; nf, na = c['n_fft'], c['angle_bins']
    a, fs = sf.read(rec['wav'], dtype='float32'); dur = len(a) / fs; at, aa = angle_track(rec['angles'], dur)
    fr = np.fft.rfftfreq(nf, 1 / fs)
    _, t, p = signal.spectrogram(a.astype(np.float64), fs=fs, window='hann', nperseg=nf, noverlap=nf - c['hop_length'], scaling='spectrum', mode='psd')
    db = 10 * np.log10(p + 1e-30); ab = np.clip((np.interp(t, at, aa) // (360 / na)).astype(int), 0, na - 1)
    r0, r1 = np.median(np.array(rec['rpm_intervals']), axis=0); q = refine_rpm(a, fs, r0, r1, dur)
    rpm = make_rpm(r0 * (1 + q[0]), r1 * (1 + q[1]), q[2], dur)(t)
    edg = np.arange(c['wind_band'][0], c['wind_band'][1] + 1, c['wind_bin_hz']); cen = (edg[:-1] + edg[1:]) / 2
    ok = ~((fr > c['pwm_line_exclude'][0]) & (fr < c['pwm_line_exclude'][1]))
    wind = np.stack([10 * np.log10(p[(fr >= lo) & (fr < hi) & ok].mean(0) + 1e-30) for lo, hi in zip(edg[:-1], edg[1:])])
    flk = (cen < c['wind_flanks'][0]) | (cen > c['wind_flanks'][1]); pm = np.vander(cen / 1000, 2) @ np.linalg.pinv(np.vander(cen[flk] / 1000, 2))
    psel = np.where((fr >= c['pwm_band'][0]) & (fr < c['pwm_band'][1]))[0][:c['pwm_bins']]
    off = np.arange(-c['whine_halfwidth_bins'], c['whine_halfwidth_bins'] + 1)
    wl = db[np.rint(c['whine_order'] * rpm / 60 / fr[1]).astype(int)[None, :] + off[:, None], np.arange(len(t))[None, :]]
    W = np.zeros((na, len(cen)), np.float32); Pm = np.zeros((na, len(psel)), np.float32); H = np.zeros((na, len(off)), np.float32)
    for i in range(na):
        m = ab == i
        if not m.any(): continue
        w = wind[:, m].mean(1); W[i] = w - pm @ w[flk]
        x = db[psel][:, m].mean(1); Pm[i] = x - np.median(x)
        h = wl[:, m].mean(1); H[i] = h - np.median(h)
    return W, Pm, H

def build_features(records):
    key = hashlib.sha256(json.dumps({k: CONFIG[k] for k in ['n_fft', 'hop_length', 'angle_bins', 'wind_band', 'wind_bin_hz', 'wind_flanks',
          'pwm_band', 'pwm_bins', 'pwm_line_exclude', 'whine_order', 'whine_halfwidth_bins', 'rpm_refine_orders']}, sort_keys=True).encode()).hexdigest()[:16]
    cache = Path(CONFIG['feature_cache_dir']) / f'features_{key}.npz'; cache.parent.mkdir(parents=True, exist_ok=True)
    if cache.exists():
        z = np.load(cache)
        if list(z['uid']) == [r['uid'] for r in records]: print('Feature cache reused:', cache.name); return [z['wind'], z['pwm'], z['whine']]
    print(f'Extracting features for {len(records)} recordings ...', flush=True)
    with ProcessPoolExecutor(CONFIG['workers']) as ex: res = list(ex.map(extract_maps, records, chunksize=8))
    maps = [np.stack([r[i] for r in res]) for i in range(3)]
    np.savez(cache, uid=np.array([r['uid'] for r in records]), wind=maps[0], pwm=maps[1], whine=maps[2]); return maps

# ==========================================
# 4. Labels
# ==========================================
def make_labels(records):
    return np.array([[[float(r['classes'][p] == c) for c in TY] for p in range(3)] for r in records], np.float32)

def labels_for(mode, Y3): return Y3 if mode == '3x3' else Y3.max(2, keepdims=True)

# ==========================================
# 5. Model Architecture
# ==========================================
class AnglePad(nn.Module):
    def __init__(self, p=1):
        super().__init__()
        self.p = p

    def forward(self, x):
        x = F.pad(x, (self.p, self.p, 0, 0))
        return F.pad(x, (0, 0, self.p, self.p), mode='circular')

    def extra_repr(self):
        return f'pad={self.p}, angle=circular, freq=zeros'

class FreqMeanMax(nn.Module):
    def forward(self, x):
        return torch.cat([x.mean(3), x.amax(3)], 1)

def make_branch():
    return nn.Sequential(
        AnglePad(1), nn.Conv2d(1, 16, kernel_size=3), nn.GroupNorm(4, 16), nn.GELU(), nn.MaxPool2d((1, 2)),
        AnglePad(1), nn.Conv2d(16, 32, kernel_size=3), nn.GroupNorm(4, 32), nn.GELU(), nn.MaxPool2d((1, 2)),
        AnglePad(1), nn.Conv2d(32, 32, kernel_size=3), nn.GroupNorm(4, 32), nn.GELU(),
        FreqMeanMax(),
    )

class FaultBandNet(nn.Module):
    MAPS = ('wind', 'pwm', 'whine')

    def __init__(self, n_types=3):
        super().__init__()
        self.n_types = n_types

        self.branches = nn.ModuleDict({m: make_branch() for m in self.MAPS})

        self.angle_conv = nn.Sequential(
            nn.Conv1d(192, 64, kernel_size=5, padding=2, padding_mode='circular'), nn.GELU(), nn.Dropout(0.3),
            nn.Conv1d(64, 64, kernel_size=5, padding=2, padding_mode='circular'), nn.GELU(),
        )

        self.head = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(64 * 36, 3 * n_types),
        )

    def forward(self, xs):
        h = torch.cat([self.branches[m](x) for m, x in zip(self.MAPS, xs)], 1)
        h = self.angle_conv(h)
        return self.head(h).view(-1, 3, self.n_types)

def describe_model(model, map_shapes):
    rows, hooks = [], []
    for name, mod in model.named_modules():
        if name and not list(mod.children()):
            hooks.append(mod.register_forward_hook(
                lambda m, i, o, name=name: rows.append((name, type(m).__name__, tuple(o.shape[1:]), sum(p.numel() for p in m.parameters())))))
    model.eval()
    with torch.no_grad():
        model([torch.zeros(1, 1, *s) for s in map_shapes])
    for h in hooks:
        h.remove()
    print(f"{'layer':<22}{'type':<12}{'output (C, 각도, 주파수)':<26}{'params':>8}")
    for name, kind, shape, n in rows:
        print(f'{name:<22}{kind:<12}{str(shape):<26}{n:>8,}')
    n_all = sum(p.numel() for p in model.parameters())
    print(f"total params {n_all:,}  (branches {sum(p.numel() for p in model.branches.parameters()):,} | "
          f"angle_conv {sum(p.numel() for p in model.angle_conv.parameters()):,} | head {sum(p.numel() for p in model.head.parameters()):,})")

def convert_legacy_state_dict(sd):
    if not any(k.startswith(('a1.', 'a2.', 'out.')) for k in sd):
        return sd
    conv_at, norm_at = [1, 6, 11], [2, 7, 12]
    top = {'a1': 'angle_conv.0', 'a2': 'angle_conv.3', 'out': 'head.2'}
    new = {}
    for k, v in sd.items():
        parts = k.split('.')
        if parts[0] == 'branches':
            at = conv_at if parts[2] == 'conv' else norm_at
            new[f"branches.{FaultBandNet.MAPS[int(parts[1])]}.{at[int(parts[3])]}.{parts[4]}"] = v
        else:
            new[f"{top[parts[0]]}.{parts[1]}"] = v
    return new

def load_pretrained(model, path):
    ck = torch.load(path, map_location='cpu')
    sd = convert_legacy_state_dict(ck.get('state_dict', ck))
    own = model.state_dict()
    ok = {k: v for k, v in sd.items() if k in own and own[k].shape == v.shape}
    model.load_state_dict(ok, strict=False)
    return sorted(set(own) - set(ok))

# ==========================================
# 6. Evaluation Functions
# ==========================================
def recording_auc(P, Y):
    v = []
    for c in range(3):
        y = Y[:, :, c].max(1)
        if 0 < y.mean() < 1: v.append(roc_auc_score(y, P[:, :, c].max(1)))
    return float(np.mean(v))

def sector_auc(P, Y):
    return float(roc_auc_score(Y.max(2).ravel(), P.max(2).ravel()))

def sector_report(Ps, Ys, fc, date, thr):
    pred = Ps >= thr; S = dict(sector_auc_all=float(roc_auc_score(Ys.ravel(), Ps.ravel())))
    m = (fc == 1) | (fc == 2); S['sector_auc_1_2'] = float(roc_auc_score(Ys[m].ravel(), Ps[m].ravel()))
    for q in range(4):
        k = fc == q; H = Ys[k] == 0; Fm = Ys[k] > 0; p = pred[k]
        S[f'fc{q}'] = dict(n_rec=int(k.sum()), healthy_sectors=int(H.sum()), healthy_correct=int((~p[H]).sum()),
                           fault_sectors=int(Fm.sum()), fault_correct=int(p[Fm].sum()), sector_acc=float((p == Fm).mean()),
                           all3_correct=float((p == Fm).all(1).mean()))
    k1 = np.where(fc == 1)[0]; S['fc1_fault_sector_ranked_first'] = float(np.mean([Ys[i, Ps[i].argmax()] for i in k1]))
    S['healthy_rec_flagged'] = {d: float(pred[(fc == 0) & (date == d)].any(1).mean()) for d in sorted(set(date[fc == 0]))}
    return S

def report(P, Y, fc, date, thr):
    S = {}
    for c, name in enumerate(TY):
        s = P[:, :, c].max(1); has = Y[:, :, c].max(1) > 0; pred = s >= thr; num = den = 0
        for b in sorted({(d, f) for d, f in zip(date, fc) if f in (1, 2)}):
            k = (date == b[0]) & (fc == b[1])
            if has[k].any() and (~has[k]).any(): w = has[k].sum() * (~has[k]).sum(); num += roc_auc_score(has[k], s[k]) * w; den += w
        one = Y[:, :, c].sum(1) == 1; h1109 = (fc == 0) & (date == '11-09'); h1102 = (fc == 0) & (date == '11-02')
        S[name] = dict(auc=float(roc_auc_score(has, s)) if 0 < has.mean() < 1 else None, in_block_auc=float(num / den) if den else None,
                       detected=float(pred[has].mean()) if has.any() else None, n=int(has.sum()),
                       fa_healthy_1109=float(pred[h1109].mean()) if h1109.any() else None, fa_healthy_1102=float(pred[h1102].mean()) if h1102.any() else None,
                       fa_fault_rec_without=float(pred[(fc > 0) & ~has].mean()),
                       position_correct=float((P[one][:, :, c].argmax(1) == Y[one][:, :, c].argmax(1)).mean()) if one.any() else None)
    pr = P.max(1) >= thr; hs = Y.max(1) > 0; anyp = pr.any(1); det = hs.any(1); k1 = (fc == 1) & det
    S['any_fault'] = dict(recall=float(anyp[det].mean()), flagged_all_healthy=float(anyp[fc == 0].mean()) if (fc == 0).any() else None,
                          flagged_only_imbalance_modulation=float(anyp[(fc > 0) & ~det].mean()) if ((fc > 0) & ~det).any() else None)
    S['type_exact_1fault'] = float((pr[k1] == hs[k1]).all(1).mean()) if k1.any() else None
    ss = P.max(2); sy = Y.max(2); m = (fc == 1) | (fc == 2)
    S['sector'] = dict(auc_within_1_2=float(roc_auc_score(sy[m].ravel(), ss[m].ravel())),
                       fc1_fault_sector_ranked_first=float(np.mean([sy[i, ss[i].argmax()] for i in np.where(k1)[0]])) if k1.any() else None,
                       all3_exact={f'fc{q}': float(((ss[fc == q] >= thr) == (sy[fc == q] > 0)).all(1).mean()) for q in range(4) if (fc == q).any()})
    return S

# ==========================================
# 7. Train Loop
# ==========================================
def finetune_checkpoint(rep):
    FT = CONFIG['finetune']
    if not FT['init_run_dir']:
        return None
    root = Path(FT['init_run_dir'])
    if root.resolve() == OUT.resolve():
        raise ValueError('finetune.init_run_dir 와 out_dir 가 같으면 시작 가중치를 덮어씀. out_dir를 바꾸세요.')
    ckpt = root / (FT['init_output_mode'] or MODE) / f"repeat{rep['repeat']:02d}" / 'faultbandnet_best.pt'
    old = {r['repeat']: r for r in json.loads((root / 'repeats.json').read_text(encoding='utf-8'))}[rep['repeat']]
    src_of = {r['uid']: r['source_files'] for r in ALL}
    seen = {s for u in old['train'] + old['validation'] for s in src_of[u]}
    test = {s for i in rep['test'] for s in RECORDS[i]['source_files']}
    if seen & test:
        raise ValueError(f"{ckpt}: 시작 가중치의 학습 원본 {len(seen & test)}개가 이번 시험과 겹침 (split 설정이 다름).")
    return ckpt

def train_fold(X, Y, tr, va, seed, run_dir, init_ckpt=None):
    device = torch.device(CONFIG['device'])
    seed_all(seed)
    FT = CONFIG['finetune'] if init_ckpt else {}
    epochs = FT.get('epochs') or CONFIG['epochs']
    lr = FT.get('learning_rate') or CONFIG['learning_rate']

    mu = [float(x[tr].mean()) for x in X]
    sd = [float(x[tr].std()) for x in X]
    Z = [torch.from_numpy(((x - m) / s).astype(np.float32))[:, None] for x, m, s in zip(X, mu, sd)]
    Yt = torch.from_numpy(Y)

    pos = Y[tr].reshape(len(tr), -1).sum(0)
    pw = torch.tensor(np.clip((len(tr) - pos) / np.maximum(pos, 1), 1, CONFIG['max_pos_weight']),
                      dtype=torch.float32, device=device).view(1, 3, -1)

    model = FaultBandNet(Y.shape[2]).to(device)
    if init_ckpt:
        skipped = load_pretrained(model, init_ckpt)
        for part in FT['freeze']:
            for p in getattr(model, part).parameters():
                p.requires_grad_(False)
        print(f"   finetune from {init_ckpt} | freeze {FT['freeze']} | new (not loaded): {skipped or 'none'}", flush=True)
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=CONFIG['weight_decay'])
    g = torch.Generator().manual_seed(seed)
    throttle = GpuThrottle(CONFIG['gpu_duty'])

    def predict(ix):
        model.eval()
        out = []
        with torch.no_grad():
            for i in range(0, len(ix), 128):
                b = ix[i:i + 128]
                out.append(torch.sigmoid(model([q[b].to(device) for q in Z])).cpu().numpy())
        return np.concatenate(out)

    val_metric = recording_auc if Y.shape[2] == 3 else sector_auc
    best = (-1, None, 0)
    history = []
    for epoch in range(1, epochs + 1):
        # [Train Phase]
        model.train()
        perm = torch.from_numpy(tr)[torch.randperm(len(tr), generator=g)]
        total = 0.0
        throttle.reset()
        for i in range(0, len(perm), CONFIG['batch_size']):
            b = perm[i:i + CONFIG['batch_size']]
            loss = F.binary_cross_entropy_with_logits(model([q[b].to(device) for q in Z]), Yt[b].to(device), pos_weight=pw)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += float(loss) * len(b)
            throttle.step()

        # [Validation Phase]
        val = val_metric(predict(va), Y[va])
        history.append(dict(epoch=epoch, train_loss=total / len(tr), val_auc=val))
        improved = val > best[0]
        if improved:
            best = (val, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}, epoch)
        if improved or epoch % CONFIG['log_every'] == 0:
            print(f"   Epoch [{epoch:02d}/{epochs}] Train - Loss: {total / len(tr):.4f} | Val - AUC: {val:.3f}"
                  + ('  * best' if improved else ''), flush=True)
        if epoch - best[2] >= CONFIG['patience']:
            break

    model.load_state_dict(best[1])
    print(f'   best epoch {best[2]}  val AUC {best[0]:.3f}', flush=True)
    torch.save(dict(state_dict=best[1], mu=mu, sd=sd, config=dict(CONFIG), epoch=best[2],
                    output_mode=MODE, init_checkpoint=str(init_ckpt) if init_ckpt else None),
               run_dir / 'faultbandnet_best.pt')
    write_json(run_dir / 'history.json', history)
    return predict

# ==========================================
# 8. Source-Disjoint Repeated Split
# ==========================================
def make_repeats(records):
    files = sorted({x for r in records for x in r['source_files']}); fi = {f: i for i, f in enumerate(files)}
    E = np.array([[fi[x] for x in r['source_files']] for r in records]); n = len(records)
    fc = FC; nf = np.bincount(fc, minlength=4); quota = np.round(CONFIG['test_rate'] * nf).astype(int)
    rng = np.random.default_rng(CONFIG['split_seed']); tested = np.zeros(n, int); reps = []
    for rep in range(CONFIG['repeats']):
        inT = np.zeros(len(files), bool); w = 1.0 / (1 + tested) ** 2; cnt = np.zeros(4, int)
        for e in rng.choice(n, n, replace=False, p=w / w.sum()):
            if (cnt >= quota).all(): break
            if cnt[fc[e]] >= quota[fc[e]] or inT[E[e]].all(): continue
            t2 = inT.copy(); t2[E[e]] = True; c2 = np.bincount(fc[t2[E].all(1)], minlength=4)
            if (c2 <= quota).all(): inT, cnt = t2, c2
        te = np.where(inT[E].all(1))[0]; pool = np.where(~inT[E].any(1))[0]
        assert not (set(E[te].ravel()) & set(E[pool].ravel()))
        frac = (np.bincount(fc[pool], minlength=4) / nf).min(); keep = np.floor(frac * nf).astype(int)
        tr, va = [], []
        for q in range(4):
            sel = rng.permutation(rng.choice(pool[fc[pool] == q], keep[q], replace=False)); nv = int(round(CONFIG['validation_fraction'] * keep[q]))
            va.append(sel[:nv]); tr.append(sel[nv:])
        tested[te] += 1; reps.append(dict(repeat=rep, train=np.sort(np.concatenate(tr)), validation=np.sort(np.concatenate(va)), test=te))
    return reps

# ==========================================
# 9. Multi-Repeat Loop Start!
# ==========================================
print('Reading dataset (read-only):', CONFIG['data_root'], flush=True)
ALL = discover(CONFIG['data_root']); X_ALL = build_features(ALL)
keep = np.array([not (set(r['classes']) & set(CONFIG['exclude_classes'])) for r in ALL])
RECORDS = [r for r, k in zip(ALL, keep) if k]; X = [x[keep] for x in X_ALL]; N = len(RECORDS)
Y3 = make_labels(RECORDS); DATE = np.array([r['date'] for r in RECORDS])
FC = Y3.max(2).sum(1).astype(int)
KIND0 = np.array(['Imb/Mod only' if f == 0 and set(r['classes']) & set(CONFIG['healthy_like']) else f'healthy {d}' for f, r, d in zip(FC, RECORDS, DATE)])
print(f'Recordings: {N} | Whine/PWM/Wind count 0/1/2/3: {np.bincount(FC).tolist()} | count-0 kinds {dict(zip(*np.unique(KIND0[FC == 0], return_counts=True)))} '
      f'| fault speakers {dict(zip(TY, Y3.sum((0, 1)).astype(int).tolist()))}')

REPEATS = make_repeats(RECORDS)
COUNT = np.zeros(N); [np.add.at(COUNT, r['test'], 1) for r in REPEATS]; TESTED = np.where(COUNT > 0)[0]
print(f'{len(REPEATS)} repeats | train {min(len(r["train"]) for r in REPEATS)}–{max(len(r["train"]) for r in REPEATS)} '
      f'| test {min(len(r["test"]) for r in REPEATS)}–{max(len(r["test"]) for r in REPEATS)} | tested at least once: {len(TESTED)}/{N}')
for q in range(4): print(f'  {q} fault: train / val / test per repeat (mean) = ' + ' / '.join(f"{np.mean([(FC[r[k]] == q).sum() for r in REPEATS]):.1f}" for k in ('train', 'validation', 'test')) + f'  of {(FC == q).sum()}')
write_json(OUT / 'repeats.json', [{'repeat': r['repeat'], **{k: [RECORDS[i]['uid'] for i in r[k]] for k in ('train', 'validation', 'test')}} for r in REPEATS])

MODE = CONFIG['output_mode']; Y = labels_for(MODE, Y3)
print(f'\nModel ({MODE}):')
describe_model(FaultBandNet(Y.shape[2]), [x.shape[1:] for x in X])
if CONFIG['train']:
    if CONFIG['device'] == 'cuda' and not torch.cuda.is_available(): raise RuntimeError('CUDA unavailable.')
    S = np.zeros(Y.shape); REP_PROBS = []
    for r in REPEATS:
        tr, va, te = r['train'], r['validation'], r['test']; run_dir = OUT / MODE / f"repeat{r['repeat']:02d}"; run_dir.mkdir(parents=True, exist_ok=True)
        print(f"[{MODE} repeat {r['repeat']}] train {len(tr)} / val {len(va)} / test {len(te)}", flush=True)
        predict = train_fold(X, Y, tr, va, CONFIG['seed_base'] + r['repeat'], run_dir, finetune_checkpoint(r))
        p = predict(te); S[te] += p; REP_PROBS.append(p)
    PROBS = (S / np.maximum(COUNT, 1)[:, None, None]).astype(np.float32)
    np.save(OUT / f'test_probabilities_{MODE}.npy', PROBS)
    np.save(OUT / f'repeat_probabilities_{MODE}.npy', np.array(REP_PROBS, dtype=object), allow_pickle=True)
else:
    PROBS = np.load(OUT / f'test_probabilities_{MODE}.npy')
    REP_PROBS = list(np.load(OUT / f'repeat_probabilities_{MODE}.npy', allow_pickle=True))
print('Training done. Raw dataset was read only.')


In [ ]:
# ==========================================
# 1. Test Evaluation
# ==========================================
THR = CONFIG['threshold']; T = TESTED; YS = Y3.max(2)
PS = PROBS.max(2)
rows = []
for i in T:
    r = RECORDS[i]
    for p, pos in enumerate(POS):
        row = dict(recording_id=r['uid'], date=r['date'], fault_count=int(FC[i]), times_tested=int(COUNT[i]), position=pos, gt_class=r['classes'][p])
        if MODE == '3x3':
            row.update({f'prob_{c}': round(float(PROBS[i, p, j]), 4) for j, c in enumerate(TY)})
            row['predicted'] = next((TY[j] for j in np.argsort(-PROBS[i, p]) if PROBS[i, p, j] >= THR), 'Healthy')
        else:
            row['prob_fault'] = round(float(PROBS[i, p, 0]), 4); row['predicted'] = 'Fault' if PROBS[i, p, 0] >= THR else 'Healthy'
        rows.append(row)
write_csv(OUT / f'test_predictions_{MODE}.csv', rows)

# ==========================================
# 2. Per-Repeat Metrics
# ==========================================
v = np.array([roc_auc_score(YS[r['test']].ravel(), p.max(2).ravel()) for r, p in zip(REPEATS, REP_PROBS) if 0 < YS[r['test']].mean() < 1])
print(f'{MODE} sector AUC per repeat: mean {v.mean():.3f} ± {v.std():.3f} (min {v.min():.3f}, max {v.max():.3f}, {len(v)} repeats)')

# ==========================================
# 3. Grand Summary
# ==========================================
SECTOR = sector_report(PS[T], YS[T], FC[T], DATE[T], THR)
DATE0 = np.where(KIND0 == 'Imb/Mod only', 'imb-mod', DATE)
TYPES = report(PROBS[T], Y3[T], FC[T], DATE0[T], THR) if MODE == '3x3' else None
write_json(OUT / f'summary_{MODE}.json', dict(config=CONFIG, n_tested=int(len(T)), times_tested_mean=float(COUNT[T].mean()), sector=SECTOR, types=TYPES))
print(f'\n=== {MODE}: 섹터 1개 정상/고장 판별 (시험된 녹음 {len(T)}개, 원본 음원 비중복) ===')
fr = lambda a, b: f'{a}/{b} = {a / b:.2f}' if b else '-'
print(f"{'fault count':<12}{'rec.':>6}{'healthy sectors right':>24}{'fault sectors right':>22}{'sector acc':>12}{'all 3 right':>13}")
for q in range(4):
    s = SECTOR[f'fc{q}']
    print(f"{q:<12}{s['n_rec']:>6}{fr(s['healthy_correct'], s['healthy_sectors']):>24}{fr(s['fault_correct'], s['fault_sectors']):>22}{s['sector_acc']:>12.2f}{s['all3_correct']:>13.2f}")
print(f"sector AUC: all {SECTOR['sector_auc_all']:.3f} | within 1-/2-fault rec. {SECTOR['sector_auc_1_2']:.3f}")
print(f"1-fault rec.: fault sector ranked 1st {SECTOR['fc1_fault_sector_ranked_first']:.3f} (random 0.33)")
flag = PS[T] >= THR; k0 = FC[T] == 0
print('count-0 recordings flagged (any sector):', {k: f"{int(flag[k0 & (KIND0[T] == k)].any(1).sum())}/{int((k0 & (KIND0[T] == k)).sum())}" for k in sorted(set(KIND0[T][k0]))})
if MODE == '3x3':
    f2 = lambda x: '  -  ' if x is None else f'{x:.2f}'
    print('\n=== 3x3: 고장 종류별 (녹음 단위) ===')
    print(f"{'fault':<7}{'AUC':>6}{'in-block':>10}{'detected':>10}{'n':>6} | false alarms: {'healthy 11/09':>14}{'healthy 11/02':>15}{'other fault rec.':>18} | position")
    for c in TY:
        s = TYPES[c]; print(f"{c:<7}{s['auc']:>6.3f}{f2(s['in_block_auc']):>10}{s['detected']:>10.2f}{s['n']:>6} | {f2(s['fa_healthy_1109']):>28}{f2(s['fa_healthy_1102']):>15}{s['fa_fault_rec_without']:>18.2f} | {f2(s['position_correct'])}")
    print('1-fault recordings, exactly the right type:', round(TYPES['type_exact_1fault'], 3))


In [ ]:
import matplotlib.pyplot as plt

# ==========================================
# 0. 데이터 준비 (Data Extraction)
# ==========================================
xs = [(q, k) for q in range(4) for k in ('healthy', 'fault') if SECTOR[f'fc{q}'][f'{k}_sectors']]
v = [SECTOR[f'fc{q}'][f'{k}_correct'] / SECTOR[f'fc{q}'][f'{k}_sectors'] for q, k in xs]

# ==========================================
# 1. 스타일 및 색상 (Nature Style)
# ==========================================
plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'], 'font.size': 9,
                     'axes.linewidth': 0.8, 'axes.spines.top': False, 'axes.spines.right': False, 'xtick.major.width': 0.8, 'ytick.major.width': 0.8})
COL = {'Whine': '#4C72B0', 'PWM': '#DD8452', 'Wind': '#55A868'}

# ==========================================
# 2. 그래프 그리기 (Plotting)
# ==========================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7.4, 2.9), gridspec_kw={'width_ratios': [1.4, 1]})

# [왼쪽: 고장 개수별 섹터 판별]
ax1.bar(range(len(xs)), v, 0.6, color=['#9ad9c5' if k == 'healthy' else '#F4A3A3' for q, k in xs], edgecolor='k', linewidth=0.6)
for a, b in enumerate(v):
    ax1.text(a, b + .02, f'{b:.2f}', ha='center', fontsize=7)
ax1.set_xticks(range(len(xs)))
ax1.set_xticklabels([f'{q} fault\n{k}' for q, k in xs], fontsize=7)
ax1.set_ylim(0, 1.1)
ax1.set_ylabel('Sectors judged correctly')
ax1.set_title(f'{MODE}: sector level, by fault count of the recording', fontsize=9)

# [오른쪽: 고장 종류별 검출·오탐 (3x3) / 섹터 지표 (3x1)]
if MODE == '3x3':
    x = np.arange(3)
    w2 = 0.26
    ax2.bar(x - w2, [TYPES[c]['detected'] for c in TY], w2, color=[COL[c] for c in TY], edgecolor='k', linewidth=0.6, label='Detected')
    ax2.bar(x, [TYPES[c]['fa_fault_rec_without'] for c in TY], w2, color=[COL[c] for c in TY], alpha=0.45, edgecolor='k', linewidth=0.6, hatch='//', label='False alarm (other-fault rec.)')
    ax2.bar(x + w2, [TYPES[c]['fa_healthy_1102'] or 0 for c in TY], w2, color='white', edgecolor='k', linewidth=0.6, label='False alarm (healthy 11/02)')
    ax2.set_xticks(x)
    ax2.set_xticklabels(TY)
    ax2.legend(frameon=False, fontsize=6, loc='upper right')
    ax2.set_title('Recording level, per fault type', fontsize=9)
else:
    vals = [SECTOR['sector_auc_1_2'], SECTOR['fc1_fault_sector_ranked_first'], SECTOR['fc1']['all3_correct'], SECTOR['fc2']['all3_correct']]
    ax2.bar(range(4), vals, 0.6, color='#8172B3', edgecolor='k', linewidth=0.6)
    ax2.axhline(1 / 3, color='gray', ls='--', lw=0.8)
    for a, b in enumerate(vals):
        ax2.text(a, b + .02, f'{b:.2f}', ha='center', fontsize=7)
    ax2.set_xticks(range(4))
    ax2.set_xticklabels(['Sector AUC\n(1-/2-fault)', 'Fault sector\nranked 1st', 'All 3 right\n(1 fault)', 'All 3 right\n(2 faults)'], fontsize=7)
    ax2.set_title('Position (sector) level', fontsize=9)
ax2.set_ylim(0, 1.1)

# ==========================================
# 3. 최종 레이아웃 정리 및 저장
# ==========================================
plt.tight_layout()
(OUT / 'figures').mkdir(exist_ok=True)
plt.savefig(OUT / f'figures/Fig_Sector_Summary_{MODE}.pdf')
plt.savefig(OUT / f'figures/Fig_Sector_Summary_{MODE}.png', dpi=200)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# ==========================================
# 0. 데이터 준비 (Data Extraction)
# ==========================================
picks = [T[np.where(FC[T] == q)[0][0]] for q in range(4) if (FC[T] == q).any()]
cols = TY if MODE == '3x3' else ['fault']
ext = [(CONFIG['wind_band'][0] / 1000, CONFIG['wind_band'][1] / 1000), (CONFIG['pwm_band'][0] / 1000, CONFIG['pwm_band'][1] / 1000), (-150, 150)]
names = ['Wind map (kHz)', 'PWM map (kHz)', 'Whine map (Hz from order 16)']

# ==========================================
# 1. 그래프 그리기 (Plotting)
# ==========================================
fig, axes = plt.subplots(len(picks), 4, figsize=(9.5, 2.2 * len(picks)), gridspec_kw={'width_ratios': [1, 1, 1, 0.9 if MODE == '3x3' else 0.5]})
for r, i in enumerate(picks):
    # [입력 지도 3장]
    for j in range(3):
        ax = axes[r, j]
        ax.imshow(X[j][i], origin='lower', aspect='auto', cmap='magma', extent=[ext[j][0], ext[j][1], 0, 360], vmin=-3, vmax=12)
        ax.set_yticks([0, 90, 180, 270, 360])
        ax.tick_params(labelsize=6)
        if r == 0:
            ax.set_title(names[j], fontsize=8)
        if j == 0:
            ax.set_ylabel('Mic angle (°)', fontsize=7)

    # [출력 확률: 위치 × 고장]
    M = PROBS[i]
    ax = axes[r, 3]
    ax.imshow(M, cmap='Reds', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(len(cols)))
    ax.set_xticklabels(cols, fontsize=6)
    ax.set_yticks(range(3))
    ax.set_yticklabels([f'{p} ({c})' for p, c in zip(POS, RECORDS[i]['classes'])], fontsize=6)
    for a in range(3):
        for b in range(len(cols)):
            ax.text(b, a, f'{M[a, b]:.2f}', ha='center', va='center', fontsize=6, color='white' if M[a, b] > 0.6 else 'black')
    if r == 0:
        ax.set_title(f'{MODE} output (true class)', fontsize=8)

# ==========================================
# 2. 최종 레이아웃 정리 및 저장
# ==========================================
plt.tight_layout()
plt.savefig(OUT / f'figures/Fig_Test_Examples_{MODE}.pdf')
plt.savefig(OUT / f'figures/Fig_Test_Examples_{MODE}.png', dpi=200)
plt.show()